# 1、clip video

In [6]:
# clip video with multiprocessing
# sudo apt update
# sudo apt install ffmpeg
import os
import ffmpeg
from tqdm import tqdm
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

sample_txt_file = "./iemocap_all_sample.txt"
data_path = "./IEMOCAP_full_release/"
video_clips_output_path = os.path.join(data_path, "MP4")

# 创建输出目录
os.makedirs(video_clips_output_path, exist_ok=True)

name_path_map = {
    "Ses01": "Session1",
    "Ses02": "Session2",
    "Ses03": "Session3",
    "Ses04": "Session4",
    "Ses05": "Session5",
}

def extract_video_segment(video_info):
    """
    处理单个视频片段的函数
    Args:
        video_info: tuple (input_path, output_path, start_time, end_time, name)
    Returns:
        tuple: (success: bool, name: str, error_msg: str)
    """
    input_path, output_path, start_time, end_time, name = video_info
    
    try:
        # 检查输入文件是否存在
        if not os.path.exists(input_path):
            return False, name, f"输入文件不存在: {input_path}"
        
        # 创建输出目录
        output_dir = os.path.dirname(output_path)
        os.makedirs(output_dir, exist_ok=True)
        
        # 执行ffmpeg命令
        (
            ffmpeg
            .input(input_path, ss=start_time, to=end_time)
            .output(output_path)
            .run(overwrite_output=True, quiet=True)
        )
        
        # 检查输出文件是否成功创建
        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            return True, name, ""
        else:
            return False, name, f"输出文件创建失败: {output_path}"
            
    except ffmpeg.Error as e:
        error_msg = f"FFmpeg错误: {e}"
        if hasattr(e, 'stderr') and e.stderr:
            stderr_str = e.stderr.decode('utf-8') if isinstance(e.stderr, bytes) else str(e.stderr)
            error_msg += f"\nFFmpeg stderr: {stderr_str}"
        return False, name, error_msg
    except Exception as e:
        return False, name, f"未知错误: {str(e)}"

def prepare_video_info_list():
    """
    准备所有需要处理的视频信息列表
    Returns:
        list: 包含所有视频处理信息的列表
    """
    video_info_list = []
    
    try:
        with open(sample_txt_file, 'r') as f:
            lines = f.readlines()
            
        print(f"读取到 {len(lines)} 行数据")
        
        for i, line in enumerate(lines):
            line = line.strip()
            
            # 跳过空行或格式错误的行
            if not line or "###" not in line:
                continue
                
            try:
                sample_info = line.split("###")[0].strip()
                items = sample_info.split(",")
                
                if len(items) < 4:
                    print(f"第{i+1}行数据不完整，跳过: {line}")
                    continue
                    
                name = items[0].strip()
                start_time = float(items[2].strip())
                end_time = float(items[3].strip())
                
                # 验证时间参数
                if start_time >= end_time:
                    print(f"时间参数错误，跳过 {name}: start={start_time}, end={end_time}")
                    continue
                
                session_name = name.split("_")[0][:5]
                
                if session_name not in name_path_map:
                    print(f"未知的会话名称，跳过: {session_name}")
                    continue
                    
                session_path = name_path_map[session_name]
                video_name = name.rsplit("_", 1)[0]
                
                input_path = os.path.join(data_path, 
                                         session_path,
                                         "dialog/avi/DivX",
                                         video_name + ".avi")
                
                output_path = os.path.join(
                    video_clips_output_path,
                    name + ".mp4"
                )
                
                video_info_list.append((input_path, output_path, start_time, end_time, name))
                
            except (ValueError, IndexError) as e:
                print(f"第{i+1}行解析错误，跳过: {e}")
                continue
                
    except FileNotFoundError:
        print(f"样本文件不存在: {sample_txt_file}")
        return []
    except Exception as e:
        print(f"读取文件时发生错误: {e}")
        return []
    
    print(f"准备处理 {len(video_info_list)} 个视频片段")
    return video_info_list

def process_videos_multiprocessing(video_info_list, max_workers=None):
    """
    使用多进程处理视频列表
    Args:
        video_info_list: 视频信息列表
        max_workers: 最大工作进程数，None表示使用CPU核心数
    """
    if not video_info_list:
        print("没有需要处理的视频")
        return
    
    # 确定工作进程数
    if max_workers is None:
        max_workers = min(mp.cpu_count(), len(video_info_list))
    
    print(f"使用 {max_workers} 个进程处理 {len(video_info_list)} 个视频片段")
    
    success_count = 0
    error_count = 0
    error_details = []
    
    start_time = time.time()
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # 提交所有任务
        future_to_info = {
            executor.submit(extract_video_segment, info): info 
            for info in video_info_list
        }
        
        # 使用tqdm显示进度
        with tqdm(total=len(video_info_list), desc="处理视频") as pbar:
            for future in as_completed(future_to_info):
                try:
                    success, name, error_msg = future.result()
                    
                    if success:
                        success_count += 1
                    else:
                        error_count += 1
                        error_details.append((name, error_msg))
                        
                except Exception as e:
                    error_count += 1
                    info = future_to_info[future]
                    error_details.append((info[4], f"进程执行错误: {e}"))
                
                pbar.update(1)
                pbar.set_postfix({
                    '成功': success_count, 
                    '失败': error_count
                })
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    # 输出统计信息
    print(f"\n处理完成:")
    print(f"总数: {len(video_info_list)}")
    print(f"成功: {success_count}")
    print(f"失败: {error_count}")
    print(f"成功率: {success_count/len(video_info_list)*100:.1f}%")
    print(f"总耗时: {elapsed_time:.2f}秒")
    print(f"平均速度: {len(video_info_list)/elapsed_time:.2f} 个/秒")
    
    # 输出错误详情（只显示前10个）
    if error_details:
        print(f"\n错误详情 (显示前10个):")
        for i, (name, error_msg) in enumerate(error_details[:10]):
            print(f"  {i+1}. {name}: {error_msg}")
        
        if len(error_details) > 10:
            print(f"  ... 还有 {len(error_details)-10} 个错误")

if __name__ == "__main__":
    # 检查ffmpeg是否可用
    import shutil
    if not shutil.which("ffmpeg"):
        print("错误: 系统中未找到ffmpeg")
        print("请安装ffmpeg后再运行此脚本")
        exit(1)
    
    # 准备视频信息列表
    video_info_list = prepare_video_info_list()
    
    if not video_info_list:
        print("没有找到需要处理的视频")
        exit(1)
    
    # 可以调整工作进程数，建议不超过CPU核心数
    max_workers = 8  # 手动指定进程数
    # max_workers = None  # 自动使用CPU核心数
    
    # 开始多进程处理
    process_videos_multiprocessing(video_info_list, max_workers)

读取到 10039 行数据
准备处理 10039 个视频片段
使用 8 个进程处理 10039 个视频片段


处理视频: 100%|██████████| 10039/10039 [1:01:23<00:00,  2.73it/s, 成功=1e+4, 失败=0]


处理完成:
总数: 10039
成功: 10039
失败: 0
成功率: 100.0%
总耗时: 3689.01秒
平均速度: 2.72 个/秒


# 2、generate wav files

In [7]:
def extract_audio_from_video(video_file, output_audio_file):
        ffmpeg.input(video_file).output(
                output_audio_file, 
                q='0', 
                map='a', 
                loglevel='quiet').run()

def process(video_files_path, wav_save_path):
    videos = [video for video in os.listdir(video_files_path) 
              if video.endswith('.mp4')]
    
    # create the save path if not exists
    if not os.path.exists(wav_save_path):
        os.makedirs(wav_save_path)

    print("Number of videos: ", len(videos))
    for video_path in tqdm(videos):
        video_path = video_path.strip()
        video_name = video_path.split('/')[-1].split('.')[0]
        wav_file_path = os.path.join(wav_save_path, video_name+'.wav')
        # force to overwrite
        if os.path.exists(wav_file_path):
            os.remove(wav_file_path)
        extract_audio_from_video(os.path.join(video_files_path, video_path), wav_file_path)

In [8]:
import sys
# video_files_path = sys.argv[1]
# wav_save_path = sys.argv[2]
video_files_path = "./IEMOCAP_full_release/MP4"
wav_save_path = "./IEMOCAP_full_release/WAV"
process(video_files_path, wav_save_path)
print("Done! check the save path: ", wav_save_path)

Number of videos:  10039


100%|██████████| 10039/10039 [12:34<00:00, 13.31it/s]

Done! check the save path:  ./IEMOCAP_full_release/WAV


# 3、extract_fbank

In [8]:
# use our process dataset

# 4、generate imgs

In [1]:
import cv2
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys  # 用于灵活解析命令行参数（保留原用法）

class VideoReader(object):
    def __init__(self, video_path):
        self.video_path = video_path
        # 初始化视频读取器，设置缓存大小减少IO等待
        self.vid = cv2.VideoCapture(self.video_path)
        self.vid.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # 减小缓存，加快帧读取响应
        # 获取视频基础信息（提前校验，避免循环内重复计算）
        self.fps = int(self.vid.get(cv2.CAP_PROP_FPS)) or 30  # 兼容异常FPS（如0）
        self.total_frames = int(self.vid.get(cv2.CAP_PROP_FRAME_COUNT))
        assert self.total_frames > 0, f"视频无法读取：{self.video_path}"

    def extract_frames(self, save_dir, frames_per_sec):
        """简化帧提取逻辑：按“每秒保留帧数”计算间隔，直接跳帧保存"""
        # 计算帧间隔（每N帧保存1帧，确保每秒保留指定数量）
        frame_interval = max(1, self.fps // frames_per_sec)  # 避免间隔为0
        frame_count = 0  # 记录当前读取的总帧数
        saved_count = 0  # 记录保存的帧数（用于校验）

        # 循环读取帧（用while True避免依赖总帧数的准确性）
        while True:
            ret, frame = self.vid.read()
            if not ret:
                break  # 读取完毕或出错，退出循环

            # 每间隔指定帧数保存1帧（跳过冗余计算）
            if frame_count % frame_interval == 0:
                # 文件名格式：00001.jpg（5位序号，确保顺序）
                save_path = os.path.join(save_dir, f"{frame_count:05d}.jpg")
                # 高效保存：imencode直接写入文件，比cv2.imwrite更快
                cv2.imencode('.jpg', frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])[1].tofile(save_path)
                saved_count += 1

            frame_count += 1

        # 释放视频资源（避免内存泄漏）
        self.vid.release()
        cv2.destroyAllWindows()

        # 校验：确保至少保存了1帧（避免空目录）
        assert saved_count > 0, f"未提取到帧：{save_dir}（视频路径：{self.video_path}）"
        return saved_count  # 返回保存的帧数，用于进度统计


class VideoProcessor(object):
    def __init__(self, videos_dir, output_dir, frames_per_sec=1, video_ext=".flv"):
        self.videos_dir = videos_dir
        self.output_dir = output_dir
        self.frames_per_sec = frames_per_sec
        self.video_ext = video_ext

        # 预处理：1. 创建输出根目录；2. 筛选所有目标视频（提前过滤，避免循环内重复判断）
        os.makedirs(self.output_dir, exist_ok=True)
        self.video_files = [
            f for f in os.listdir(self.videos_dir) 
            if f.endswith(self.video_ext) and os.path.isfile(os.path.join(self.videos_dir, f))
        ]

        # 校验：确保有视频可处理
        assert len(self.video_files) > 0, f"在{self.videos_dir}中未找到{self.video_ext}格式的视频"
        print(f"发现视频数量：{len(self.video_files)} | 每秒保留帧数：{self.frames_per_sec}")

    def process_single_video(self, video_filename):
        """单视频处理函数（供多线程调用）"""
        try:
            # 1. 构建路径：视频路径 + 输出子目录（按视频名创建）
            video_path = os.path.join(self.videos_dir, video_filename)
            video_name = os.path.splitext(video_filename)[0]  # 去掉后缀（如"video1.flv"→"video1"）
            save_dir = os.path.join(self.output_dir, video_name)
            os.makedirs(save_dir, exist_ok=True)  # 线程安全：exist_ok=True避免目录冲突

            # 2. 提取帧
            reader = VideoReader(video_path)
            saved_frames = reader.extract_frames(save_dir, self.frames_per_sec)
            
            # 3. 返回成功信息（用于进度条更新）
            return (True, f"成功：{video_filename} | 保存帧数：{saved_frames}")
        
        except Exception as e:
            # 捕获异常：跳过错误视频，继续处理其他视频（避免整体中断）
            return (False, f"失败：{video_filename} | 错误原因：{str(e)[:100]}")  # 截断长错误信息

    def process_all_videos(self, max_workers=4):
        """多线程批量处理所有视频（核心优化点）"""
        print(f"开始批量处理（线程数：{max_workers}）...")
        results = []

        # 1. 创建线程池：IO密集型任务用ThreadPoolExecutor（比多进程更轻量、速度更快）
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # 2. 提交所有视频处理任务（非阻塞）
            tasks = {executor.submit(self.process_single_video, fn): fn for fn in self.video_files}
            
            # 3. 实时更新进度条（as_completed获取已完成任务）
            for future in tqdm(as_completed(tasks), total=len(tasks), desc="处理进度"):
                success, msg = future.result()
                results.append((success, msg))
                # 打印失败信息（成功信息可省略，避免刷屏）
                if not success:
                    print(f"\n{msg}")

        # 4. 统计结果
        success_count = sum(1 for s, _ in results if s)
        fail_count = len(results) - success_count
        print(f"\n处理完成！成功：{success_count} 个 | 失败：{fail_count} 个")
        print(f"输出目录：{os.path.abspath(self.output_dir)}")


if __name__ == "__main__":
    # 兼容原命令行用法 + 支持硬编码（灵活切换）
    if len(sys.argv) == 5:
        # 命令行模式：python video_preprocessing.py <videos_path> <output_path> <frames_per_sec> <video_type>
        videos_dir = sys.argv[1]
        output_dir = sys.argv[2]
        frames_per_sec = int(sys.argv[3])
        video_ext = sys.argv[4]
    else:
        # 硬编码模式（调试/快速运行用）
        videos_dir = "./IEMOCAP_full_release/MP4"
        output_dir = "./IEMOCAP_full_release/IMAGE_KEPT_2_PER_SEC"
        frames_per_sec = 2
        video_ext = ".mp4"

    # 初始化处理器并执行（线程数建议：CPU核心数的1-2倍，如4/8）
    processor = VideoProcessor(
        videos_dir=videos_dir,
        output_dir=output_dir,
        frames_per_sec=frames_per_sec,
        video_ext=video_ext
    )
    processor.process_all_videos(max_workers=8)  # 根据CPU核心数调整max_workers（如8核设为8）

发现视频数量：10039 | 每秒保留帧数：2
开始批量处理（线程数：8）...


处理进度: 100%|██████████| 10039/10039 [04:29<00:00, 37.26it/s]


处理完成！成功：10039 个 | 失败：0 个
输出目录：/root/autodl-tmp/AMRe/data/Dataset/IEMOCAP/IEMOCAP_full_release/IMAGE_KEPT_2_PER_SEC


# 5、generate text token

In [11]:
import os
import torch
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

sample_txt_file = "./iemocap_all_sample.txt"
output_dir = "./IEMOCAP_full_release/text_token"
os.makedirs(output_dir, exist_ok=True)

# 指定 tokenizer 路径或名称
TOKENIZER_PATH = "/root/autodl-tmp/AMRe/model/bert-base-uncased"
MAX_LENGTH = 128
NUM_WORKERS = 8  # None 表示自动使用 CPU 核心数
# NUM_WORKERS = None  # None 表示自动使用 CPU 核心数

# 全局变量（在 worker 进程中初始化）
_TOKENIZER = None
_MAX_LENGTH = None

def _worker_init(tokenizer_path, max_length):
    """在每个子进程中执行一次的初始化函数：加载 tokenizer"""
    global _TOKENIZER, _MAX_LENGTH
    from transformers import BertTokenizer
    _TOKENIZER = BertTokenizer.from_pretrained(tokenizer_path)
    _MAX_LENGTH = max_length

def _tokenize_and_save(job):
    """
    job: (sample_name, text, output_dir)
    返回: (sample_name, success_bool, errmsg)
    """
    global _TOKENIZER, _MAX_LENGTH
    try:
        sample_name, text, out_dir = job
        if _TOKENIZER is None:
            # 容错：若进程未通过 initializer 启动，再次加载
            from transformers import BertTokenizer
            _TOKENIZER = BertTokenizer.from_pretrained(TOKENIZER_PATH)
            _MAX_LENGTH = MAX_LENGTH

        encoded = _TOKENIZER(
            text,
            padding="max_length",
            truncation=True,
            max_length=_MAX_LENGTH,
            return_tensors="pt"
        )

        tokens = encoded["input_ids"][0]
        attention_mask = encoded["attention_mask"][0].float()

        token_file = os.path.join(out_dir, f"{sample_name}_token.pt")
        pm_file = os.path.join(out_dir, f"{sample_name}_pm.pt")

        # 原子写入：先写 tmp 再重命名（减少并发冲突风险）
        torch.save(tokens, token_file + ".tmp")
        os.replace(token_file + ".tmp", token_file)

        torch.save(attention_mask, pm_file + ".tmp")
        os.replace(pm_file + ".tmp", pm_file)

        return sample_name, True, ""
    except Exception as e:
        return (sample_name if 'sample_name' in locals() else "unknown", False, str(e))


def prepare_jobs(sample_txt_file, out_dir):
    jobs = []
    if not os.path.exists(sample_txt_file):
        print(f"样本文件不存在: {sample_txt_file}")
        return jobs

    with open(sample_txt_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "###" not in line:
                continue
            meta, text = line.split("###", 1)
            sample_name = meta.split(",")[0].strip()
            text = text.strip()
            if not sample_name or not text:
                continue
            jobs.append((sample_name, text, out_dir))
    return jobs


def main():
    import multiprocessing
    jobs = prepare_jobs(sample_txt_file, output_dir)
    if not jobs:
        print("没有待处理样本。")
        return

    max_workers = NUM_WORKERS or min(multiprocessing.cpu_count(), len(jobs))
    print(f"准备处理 {len(jobs)} 条样本，使用 {max_workers} 进程")

    success = 0
    failed = 0
    errors = []

    with ProcessPoolExecutor(max_workers=max_workers, initializer=_worker_init, initargs=(TOKENIZER_PATH, MAX_LENGTH)) as exe:
        futures = {exe.submit(_tokenize_and_save, job): job[0] for job in jobs}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="tokenizing"):
            try:
                name, ok, msg = fut.result()
                if ok:
                    success += 1
                else:
                    failed += 1
                    errors.append((name, msg))
            except Exception as e:
                failed += 1
                errors.append((futures.get(fut, "unknown"), str(e)))

    print(f"\n完成: 成功 {success}, 失败 {failed}")
    if errors:
        print("部分错误示例（最多显示10条）：")
        for i, (n, m) in enumerate(errors[:10], 1):
            print(f"{i}. {n}: {m}")



main()

准备处理 10039 条样本，使用 8 进程


/root/miniconda3/envs/AMRe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/AMRe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/AMRe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/AMRe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from


完成: 成功 10039, 失败 0


In [7]:
import os
import torch  # Import PyTorch
from transformers import BertTokenizer

# Path to the file that contains all sample information
sample_txt_file = "./iemocap_all_sample.txt"

# Output directory where the token and padding mask files will be saved
output_dir = "./IEMOCAP_full_release/text_token"

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("/root/autodl-tmp/AMRe/model/bert-base-uncased")

# Open the sample file and process it line by line
with open(sample_txt_file, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # Skip empty lines

        # Each line is assumed to be in the format:
        #   Ses05F_impro07_F013, exc, 60.2400, 64.5700 ### I know they have a really, really great program for what I want to do.
        # We split the line on "###" to separate the metadata from the text.
        if "###" not in line:
            continue  # Skip lines that do not contain the separator

        meta, text = line.split("###", 1)
        text = text.strip()  # This is the text to tokenize

        # Extract the sample name from the metadata (it's the first item before the first comma)
        sample_name = meta.split(",")[0].strip()

        # Tokenize the text.
        # Using return_tensors="pt" returns PyTorch tensors directly.

        # print("text: ", text)
        # print("sample_name: ", sample_name)
            
        encoded = tokenizer(
            text,
            padding="max_length",  # Pad to the maximum sequence length
            truncation=True,
            max_length=128,  # Max sequence length
            return_tensors="pt"  # Returns PyTorch tensors
        )

        # Extract token ids and attention mask from the encoded output.
        # The tokenizer returns tensors with a batch dimension, so we take the first (and only) element.
        tokens = encoded["input_ids"][0]
        attention_mask = encoded["attention_mask"][0].float()  # Convert to float if needed

        # Create the padding mask.
        # Here we assume that positions with a value 0 in the attention mask are padded,
        # so subtracting the attention mask from 1 gives a mask with 1.0 at padded positions.
        #padding_mask = 1.0 - attention_mask

        # Create file names based on the sample name with a .pt extension.
        token_file = os.path.join(output_dir, f"{sample_name}_token.pt")
        pm_file = os.path.join(output_dir, f"{sample_name}_pm.pt")

        # Save the token IDs and padding mask as .pt files.
        torch.save(tokens, token_file)
        torch.save(attention_mask, pm_file)

        # print(f"Processed sample: {sample_name}")